In [2]:
import numpy as np
import pandas as pd
import os
import ast
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")


# Load Our Best Projection Model

In [93]:
df = pd.read_csv("retrieval_projection_results/retrieval_results.csv")

# Convert stringified dicts → real dicts
for col in ["i2t", "t2i", "m2t", "m2i", "eff_i2t", "eff_t2i", "eff_m2t", "eff_m2i"]:
    df[col] = df[col].apply(ast.literal_eval)

# Extract R@1
df["i2t_R@1"] = df["i2t"].apply(lambda d: d["R@1"])
df["t2i_R@1"] = df["t2i"].apply(lambda d: d["R@1"])

# Remove invalid projection rows
df = df[df["projection"] != "random"]

print(df.head())

# Keep only rows with valid embeddings
valid_rows = []
for idx, row in df.iterrows():
    v, t, proj = row["vision_model"], row["text_model"], row["projection"]
    Xv_path = f"projected_embeddings/{v}_{t}_{proj}_Xv.npy"
    Xt_path = f"projected_embeddings/{v}_{t}_{proj}_Xt.npy"
    if os.path.exists(Xv_path) and os.path.exists(Xt_path):
        Xv = np.load(Xv_path)
        Xt = np.load(Xt_path)
        if Xv.shape[0] == 8091 and Xt.shape[0] == 40455:
            valid_rows.append(idx)

df = df.loc[valid_rows]

best_i2t_cfg = df.loc[df["i2t_R@1"].idxmax()]
best_t2i_cfg = df.loc[df["t2i_R@1"].idxmax()]

best_i2t_cfg, best_t2i_cfg


   vision_model text_model projection  fusion  \
4  mobilenet_v3    roberta       wcca  concat   
5  mobilenet_v3    roberta       wcca     add   
6  mobilenet_v3    roberta       wcca   gated   
7  mobilenet_v3    roberta       wcca     mul   
8  mobilenet_v3    roberta       cpca  concat   

                                                 i2t  \
4  {'R@1': 0.012235817575083427, 'R@5': 0.0461006...   
5  {'R@1': 0.012235817575083427, 'R@5': 0.0461006...   
6  {'R@1': 0.012235817575083427, 'R@5': 0.0461006...   
7  {'R@1': 0.012235817575083427, 'R@5': 0.0461006...   
8  {'R@1': 0.005314547027561488, 'R@5': 0.0184155...   

                                                 t2i  \
4  {'R@1': 0.0004449388209121246, 'R@5': 0.001903...   
5  {'R@1': 0.0004449388209121246, 'R@5': 0.001903...   
6  {'R@1': 0.0004449388209121246, 'R@5': 0.001903...   
7  {'R@1': 0.0004449388209121246, 'R@5': 0.001903...   
8  {'R@1': 0.0011123470522803114, 'R@5': 0.003287...   

                               

(vision_model                                         mobilenet_v3
 text_model                                                   bert
 projection                                                   wcca
 fusion                                                     concat
 i2t             {'R@1': 0.014089729328883945, 'R@5': 0.0507971...
 t2i             {'R@1': 0.007934742306266222, 'R@5': 0.0278333...
 m2t             {'R@1': 0.0, 'R@5': 0.0, 'R@10': 0.0, 'MedR': ...
 m2i             {'R@1': 0.00012359411692003462, 'R@5': 0.00049...
 eff_i2t         {'latency': 0.9125430583953857, 'per_sample': ...
 eff_t2i         {'latency': 1.0014292240142821, 'per_sample': ...
 eff_m2t         {'latency': 1.909508228302002, 'per_sample': 0...
 eff_m2i         {'latency': 0.3922530174255371, 'per_sample': ...
 i2t_R@1                                                   0.01409
 t2i_R@1                                                  0.007935
 Name: 16, dtype: object,
 vision_model                       

# Load Projected Embeddings

In [4]:
def load_projected_embeddings(cfg):
    v = cfg["vision_model"]
    t = cfg["text_model"]
    proj = cfg["projection"]
    Xv = np.load(f"projected_embeddings/{v}_{t}_{proj}_Xv.npy")
    Xt = np.load(f"projected_embeddings/{v}_{t}_{proj}_Xt.npy")
    return Xv, Xt

Xv_best_i2t, Xt_best_i2t = load_projected_embeddings(best_i2t_cfg)
Xv_best_t2i, Xt_best_t2i = load_projected_embeddings(best_t2i_cfg)

# Load CLIP + OpenCLIP embeddings
clip_vision = np.load("TFE_Data/Results_Multimodal/Flickr8k/clip/vision_embeddings.npy")
clip_text   = np.load("TFE_Data/Results_Multimodal/Flickr8k/clip/text_embeddings.npy")

openclip_vision = np.load("TFE_Data/Results_Multimodal/Flickr8k/openclip_l14/vision_embeddings.npy")
openclip_text   = np.load("TFE_Data/Results_Multimodal/Flickr8k/openclip_l14/text_embeddings.npy")


In [5]:
# 8091 images, 40455 captions
gt_i2t = {i: list(range(i*5, i*5 + 5)) for i in range(8091)}
gt_t2i = {j: j // 5 for j in range(40455)}


# Retrieval Metrics

In [6]:
def recall_at_k_i2t(sim_matrix, k):
    correct = 0
    for i in range(sim_matrix.shape[0]):
        top_k = np.argsort(sim_matrix[i])[::-1][:k]
        if any(c in top_k for c in gt_i2t[i]):
            correct += 1
    return correct / sim_matrix.shape[0]

def recall_at_k_t2i(sim_matrix, k):
    correct = 0
    for j in range(sim_matrix.shape[0]):
        top_k = np.argsort(sim_matrix[j])[::-1][:k]
        if gt_t2i[j] in top_k:
            correct += 1
    return correct / sim_matrix.shape[0]


# Unified Evaluation

In [7]:
def evaluate_flickr8k(Xv, Xt):
    sims_i2t = cosine_similarity(Xv, Xt)
    sims_t2i = cosine_similarity(Xt, Xv)

    return {
        "i2t_R@1":  recall_at_k_i2t(sims_i2t, 1),
        "i2t_R@5":  recall_at_k_i2t(sims_i2t, 5),
        "i2t_R@10": recall_at_k_i2t(sims_i2t, 10),
        "i2t_R@50": recall_at_k_i2t(sims_i2t, 50),

        "t2i_R@1":  recall_at_k_t2i(sims_t2i, 1),
        "t2i_R@5":  recall_at_k_t2i(sims_t2i, 5),
        "t2i_R@10": recall_at_k_t2i(sims_t2i, 10),
        "t2i_R@50": recall_at_k_t2i(sims_t2i, 50),
    }


In [8]:
results_best_i2t = evaluate_flickr8k(Xv_best_i2t, Xt_best_i2t)
results_best_t2i = evaluate_flickr8k(Xv_best_t2i, Xt_best_t2i)
results_clip     = evaluate_flickr8k(clip_vision, clip_text)
results_openclip = evaluate_flickr8k(openclip_vision, openclip_text)

results_best_i2t, results_best_t2i, results_clip, results_openclip


({'i2t_R@1': 0.07526881720430108,
  'i2t_R@5': 0.19985168705969597,
  'i2t_R@10': 0.28587319243604004,
  'i2t_R@50': 0.5483870967741935,
  't2i_R@1': 0.007934742306266222,
  't2i_R@5': 0.027833395130391795,
  't2i_R@10': 0.0457051044370288,
  't2i_R@50': 0.12836484983314794},
 {'i2t_R@1': 0.07526881720430108,
  'i2t_R@5': 0.19985168705969597,
  'i2t_R@10': 0.28587319243604004,
  'i2t_R@50': 0.5483870967741935,
  't2i_R@1': 0.007934742306266222,
  't2i_R@5': 0.027833395130391795,
  't2i_R@10': 0.0457051044370288,
  't2i_R@50': 0.12836484983314794},
 {'i2t_R@1': 0.47435422073909284,
  'i2t_R@5': 0.7049808429118773,
  'i2t_R@10': 0.7921146953405018,
  'i2t_R@50': 0.9379557533061427,
  't2i_R@1': 0.29823260412804353,
  't2i_R@5': 0.5330861450994933,
  't2i_R@10': 0.6328018786305771,
  't2i_R@50': 0.8439500679767643},
 {'i2t_R@1': 0.5438141144481523,
  'i2t_R@5': 0.7706093189964157,
  'i2t_R@10': 0.8497095538252379,
  'i2t_R@50': 0.9605734767025089,
  't2i_R@1': 0.3612408849338771,
  't2i_R

In [9]:
comparison = pd.DataFrame([
    {"model": "Our Best i2t", **results_best_i2t},
    {"model": "Our Best t2i", **results_best_t2i},
    {"model": "CLIP ViT-B/32", **results_clip},
    {"model": "OpenCLIP ViT-L/14", **results_openclip},
])

comparison


,model,i2t_R@1,i2t_R@5,i2t_R@10,i2t_R@50,t2i_R@1,t2i_R@5,t2i_R@10,t2i_R@50
0,Our Best i2t,0.075269,0.199852,0.285873,0.548387,0.007935,0.027833,0.045705,0.128365
1,Our Best t2i,0.075269,0.199852,0.285873,0.548387,0.007935,0.027833,0.045705,0.128365
2,CLIP ViT-B/32,0.474354,0.704981,0.792115,0.937956,0.298233,0.533086,0.632802,0.843950
3,OpenCLIP ViT-L/14,0.543814,0.770609,0.849710,0.960573,0.361241,0.602694,0.699320,0.881745


In [10]:
def benchmark_retrieval(Q, I, n_runs=5):
    Q_t = torch.tensor(Q, device="cuda")
    I_t = torch.tensor(I, device="cuda")

    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    latencies = []
    for _ in range(n_runs):
        t0 = time.time()
        _ = Q_t @ I_t.T
        torch.cuda.synchronize()
        latencies.append(time.time() - t0)

    avg = np.mean(latencies)
    return {
        "latency": avg,
        "throughput": len(Q) / avg,
        "gpu_mem": torch.cuda.max_memory_allocated() / (1024**2)
    }


In [11]:
eff_best_i2t = benchmark_retrieval(Xv_best_i2t, Xt_best_i2t)
eff_clip     = benchmark_retrieval(clip_vision, clip_text)
eff_openclip = benchmark_retrieval(openclip_vision, openclip_text)


NameError: name 'torch' is not defined

In [ ]:
comparison = pd.DataFrame([
    {"model": "MobileNetV3-BERT-WCCA", **results_best_i2t, **eff_best_i2t},
    {"model": "CLIP ViT-B/32", **results_clip, **eff_clip},
    {"model": "OpenCLIP ViT-L/14", **results_openclip, **eff_openclip},
])

comparison


,model,i2t_R@1,i2t_R@5,i2t_R@10,i2t_R@50,t2i_R@1,t2i_R@5,t2i_R@10,t2i_R@50,latency,throughput,gpu_mem
0,MobileNetV3-BERT-WCCA,0.075269,0.199852,0.285873,0.548387,0.007935,0.027833,0.045705,0.128365,0.008983,900687.766442,1280.708008
1,CLIP ViT-B/32,0.474354,0.704981,0.792115,0.937956,0.298233,0.533086,0.632802,0.843950,0.016731,483581.616436,1351.573730
2,OpenCLIP ViT-L/14,0.543814,0.770609,0.849710,0.960573,0.361241,0.602694,0.699320,0.881745,0.024918,324706.483621,1398.981934
